In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

#### MNIST dataset

In [2]:
# Function to one-hot encode the target variable into the 10 classes (0-9)
# Input shape: (N,),    Output: (N, 10)
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, 10))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y

In [3]:
# Loading the MNIST dataset
train_data=pd.read_csv(r"./mnist_train.csv")
test_data=pd.read_csv(r"./mnist_test.csv")

# Preprocessing the data
train_data=train_data.to_numpy()    # train_data shape: (60000, 785)
test_data=test_data.to_numpy()      # test_data shape: (10000, 785)

X_train=train_data[:,1:]            # X_train shape: (60000, 784)
y_train=train_data[:,0]             # y_train shape: (60000,)
X_test=test_data[:,1:]              # X_test shape: (10000, 784)
y_test=test_data[:,0]               # y_test shape: (10000,)

X_train = X_train / 255.0           # Normalizing the data
X_test = X_test / 255.0

one_hot_y_train = one_hot(y_train)  # one_hot_y_train shape: (60000, 10)
one_hot_y_test = one_hot(y_test)    # one_hot_y_test shape: (10000, 10)

In [4]:
class Network:
    def __init__(self, arch: list, x_train, y_train, activation_functions):
        """
        train: (datapoints, inputs)
        w: (layer-1, input, output)
        x: (layer, datapoints, inputs)
        """
        self.arch = arch
        self.layers = len(arch) - 1
        self.train = x_train
        self.examples = self.train.shape[0]
        # self.train = np.append(self.train,np.ones((self.train.shape[0],1)),axis=1)
        
        self.target = y_train
        self.x = np.ndarray([])
        self.deltas = [None for i in range(self.layers)]
        self.activation = activation_functions
        
        self.w = []
        for i in range(0,len(arch)-1):
            # self.w.append(np.random.random((arch[i]+1,arch[i+1]))-0.5)
            self.w.append(np.random.rand(arch[i]+1,arch[i+1])-0.5)

    def target(self, l: int):
        if l == self.layers - 1:
            return self.target
        else:
            return output(l)
        
    def output(self,l: int):
        return self.activation[l](np.append(self.x[l],-1 * np.ones((self.x[l].shape[0],1)),axis=1) @ self.w[l])

    def forward(self):
        self.x = [self.train]
        for i in range(1,self.layers):
            self.x.append(self.output(i-1))

    def backward(self, lr: float, loss="ce"):
        for l in reversed(range(0,self.layers)):
            if l == self.layers-1:

                if loss=="tss":
                    del_j = (self.target - self.output(l)) * self.output(l) * (1-self.output(l))
                if loss=="ce":
                    del_j = (self.target - self.output(l))

            else:
                # del_j = np.sum(self.w[l+1][None,:,:]*self.deltas[l+1][:,None,:],axis=2)
                del_j = self.deltas[l+1]@self.w[l+1][:-1,:].T
                if loss=="tss":
                    del_j = del_j * self.x[l+1] * (1-self.x[l+1])
                if loss=="ce":
                    del_j = del_j

            self.deltas[l] = del_j

        # Update Weights
        for l in range(self.layers):
            # self.w[l] -= lr * np.mean(self.x[l][:,:,None] * self.deltas[l][:,None,:],axis=0)
            del_w = lr * np.append(self.x[l],-1 * np.ones((self.examples,1)),axis=1).T @ self.deltas[l] / (self.x[l].shape[0] +1)
            # print(del_w)
            self.w[l] -= del_w
    
    def get_accuracy(self,y, y_pred):
        return np.sum(np.argmax(y,axis=1) == np.argmax(y_pred,axis=1))/self.examples

    def cross_entropy_loss(self,Y, Y_hat):
        return -np.mean(Y*np.log(Y_hat)+(1-Y)*np.log(1-Y_hat))
        # return Y*np.log(Y_hat)
    
    def train_ffnn(self,epochs: int, lr: float):
        losses = []
        for epoch in range(epochs):
            # time_start = time.time()
            self.forward()
            y_pred = self.output(self.layers-1)
            # time_end1 = time.time()
            self.backward(lr)
            # time_end2 = time.time()
            
            accuracy = self.get_accuracy(self.target, y_pred)
            loss = self.cross_entropy_loss(self.target,y_pred)
            losses.append(loss)
            if (epoch %20 == 0):
                # print(f"Epoch: {epoch}, time: {time_end2 - time_start}s, loss: {loss}, accuracy: {accuracy}")
                print(f"Epoch: {epoch}, loss: {loss}, accuracy: {accuracy}")

        plt.plot(losses)
        
    def evaluate(self, x_test):
        self.x = [x_test]
        for i in range(1,self.layers+1):
            self.x.append(self.output(i-1))
        return self.x[-1]
    
    def test(self,x_test,y_test):
        y_pred = self.evaluate(x_test)

        accuracy = self.get_accuracy(y_test,y_pred)
        loss = self.cross_entropy_loss(y_test,y_pred)
        print(f"LOSS: {loss}, accuracy: {accuracy}")

In [7]:
def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    Z_new = Z - np.max(Z, axis=0, keepdims=True)
    A = np.exp(Z_new) / np.sum(np.exp(Z_new), axis=0, keepdims=True)
    return A

def sigmoid(Z):
    return 1/(1+np.exp(-Z))

my_network = Network([X_train.shape[1],20,10],X_train, one_hot_y_train, [ReLU,softmax])

In [8]:
my_network.train_ffnn(500,0.5)

Epoch: 0, loss: 1.2915814251618452, accuracy: 0.09688333333333334


/tmp/ipykernel_21694/1133438675.py:68: RuntimeWarning: divide by zero encountered in log
  return -np.mean(Y*np.log(Y_hat)+(1-Y)*np.log(1-Y_hat))
/tmp/ipykernel_21694/1133438675.py:68: RuntimeWarning: invalid value encountered in multiply
  return -np.mean(Y*np.log(Y_hat)+(1-Y)*np.log(1-Y_hat))


Epoch: 20, loss: nan, accuracy: 0.09871666666666666
Epoch: 40, loss: nan, accuracy: 0.09871666666666666
Epoch: 60, loss: nan, accuracy: 0.09871666666666666
Epoch: 80, loss: nan, accuracy: 0.09871666666666666


KeyboardInterrupt: 

In [ ]:
my_network.test(X_test,one_hot_y_test)

#### Training on the other Boolean functions